In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:

# Task 1: Write your code here:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error



p = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(p)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')
  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df =df.drop(columns=["Order_ID"],axis=1)
df.head()

In [ ]:
# Task 2: Write your code here:
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)
print(df.shape)
df = df.dropna()
print(df.shape)
# [1372 rows x 8 columns]

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
print(df.shape)
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)


In [ ]:
df.shape

In [ ]:
X = df.drop(columns=["Delivery_Time"],axis=1)
y = df["Delivery_Time"]
X.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
print(categorical_cols)
# one = OneHotEncoder(sparse_output=False)
# X_encoded = pd.DataFrame(one.fit_transform(df[categorical_cols]),columns=one.get_feature_names_out(categorical_cols))
# X= X.drop(columns=categorical_cols)
# X =  pd.DataFrame(X,X_encoded)
# X_encoded.info()
from sklearn.preprocessing import LabelEncoder
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
df

In [ ]:
# Task 5: Write your code here:
# Scale features - fit on train, transform both
scaler = StandardScaler()
df = scaler.fit_transform(df)
print(df)

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
LE = LabelEncoder()
y_encode = LE.fit_transform(y)

In [ ]:
# Task 6: Write your code here:


In [ ]:
# Task 1: Write your code here:
# print(X_scaled)
X = df.drop(columns=["delivery_time"])
y = df["delivery_time"]


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error

# Task 2,3,4,5: Write your code here:
RF = RandomForestRegressor(n_estimators=10)
n_splits=5
skfold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
averg = 0

for fold_idx, (train_index, test_index) in enumerate(skfold.split(X_train, y_train)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  RF.fit(X_train, y_train)
  y_pred = RF.predict(X_test)
  mae = mean_absolute_error(y_test, y_pred)
  averg += mae
print(averg / n_splits)



In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': RF.feature_names_in_,
    'importance': RF.feature_importances_
}).sort_values('importance', ascending=False)
# print(RF.feature_names_in_)
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
print(plt.hist(y_pred, bins=50))


In [ ]:
!pip install catboost


In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=10),
  "CatBoost": CatBoostRegressor(verbose=0)
}
all_results = {}

for name in models:
  all_results[name] = {"mae": []}

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X_train, y_train)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)

    all_results[model_name]["mae"].append(mae)

for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MAE:  {np.mean(all_results[model_name]['mse']):.4f}")

